In [ ]:
# %% [markdown]
# # Baseline: Multi-output regression of lumbar disc coordinates (PyTorch)
# Predict (relative_x, relative_y) for L1/2..L5/S1 -> 10 outputs.

# %%
import os
from pathlib import Path
import random
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

# ---- Paths ----
ROOT = Path(r"C:/Users/hyeon/Documents/miniconda_medimg_env/data/lumbar-coordinate")
DATA_DIR = ROOT / "data"
CSV_PRETRAIN = ROOT / "coords_pretrain.csv"
OUT_DIR = ROOT / "_runs_baseline"
OUT_DIR.mkdir(exist_ok=True, parents=True)

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# ---- Labels & ordering ----
LEVELS = ["L1/L2","L2/L3","L3/L4","L4/L5","L5/S1"]
LEVEL2IDX = {lvl:i for i,lvl in enumerate(LEVELS)}

# ---- Dataset ----
class LumbarDataset(Dataset):
    def __init__(self, df, image_root, transform=None, allow_missing=False, image_map=None):
        self.df = df
        self.image_root = image_root
        self.transform = transform
        self.allow_missing = allow_missing
        self.image_map = image_map or self._build_image_map(image_root)

        # Group rows by filename and build 10-dim targets in LEVEL order
        self.samples = []
        for fn, g in df.groupby("filename"):
            fn_key = fn.strip().lower()
            base_key = Path(fn).name.lower()
            img_path = self.image_map.get(fn_key) or self.image_map.get(base_key)
            if img_path is None:
                if allow_missing:
                    continue
                else:
                    raise FileNotFoundError(f"Missing image for {fn}")
            # Initialize with NaNs and fill per level
            y = np.full((len(LEVELS), 2), np.nan, dtype=np.float32)
            for _, row in g.iterrows():
                idx = LEVEL2IDX.get(row["level"])
                if idx is not None:
                    y[idx,0] = float(row["relative_x"])
                    y[idx,1] = float(row["relative_y"])
            if np.isnan(y).any():
                # Skip if any level missing
                continue
            self.samples.append((str(fn), str(img_path), y.reshape(-1)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples constructed. Check CSV/paths.")

    def _build_image_map(self, root):
        exts = {".jpg",".jpeg",".png",".bmp"}
        mp = {}
        for r, d, files in os.walk(root):
            for f in files:
                if Path(f).suffix.lower() in exts:
                    abs_p = Path(r) / f
                    rel = abs_p.relative_to(root).as_posix()
                    mp[rel.lower()] = abs_p
                    mp[f.lower()] = abs_p
        return mp

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fn, img_path, target = self.samples[idx]
        im = Image.open(img_path).convert("RGB")
        if self.transform:
            im = self.transform(im)
        target = torch.from_numpy(target.copy())  # shape (10,)
        return im, target, fn

# ---- Transforms ----
IMG_SIZE = 320
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomRotation(degrees=7, fill=(0,0,0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

# ---- Load CSV and split ----
df = pd.read_csv(CSV_PRETRAIN)
# Keep only expected columns & levels, assert completeness
df = df[["filename","level","relative_x","relative_y","source"]]
df = df[df["level"].isin(LEVELS)].copy()

# Stratify by source to reduce domain leakage
# Build image-level dataframe
img_level = (df.groupby(["filename","source"])["level"]
               .nunique()
               .reset_index()
               .rename(columns={"level":"n_levels"}))
img_level = img_level[img_level["n_levels"]==5].reset_index(drop=True)

# deterministic split
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
# Use first split for train/val
fold = 0
(train_idx, val_idx) = next(sgkf.split(img_level, img_level["source"], groups=img_level["source"]))
train_files = set(img_level.loc[train_idx,"filename"])
val_files   = set(img_level.loc[val_idx,"filename"])

df_train = df[df["filename"].isin(train_files)].copy()
df_val   = df[df["filename"].isin(val_files)].copy()

print(f"Train images: {df_train['filename'].nunique()}, Val images: {df_val['filename'].nunique()}")

train_ds = LumbarDataset(df_train, DATA_DIR, transform=train_tfms, allow_missing=True)
val_ds   = LumbarDataset(df_val,   DATA_DIR, transform=val_tfms,   allow_missing=True)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# ---- Model ----
backbone = torchvision.models.resnet18(weights="IMAGENET1K_V1")
# Replace the final FC
in_features = backbone.fc.in_features
backbone.fc = nn.Linear(in_features, 10)  # 5 levels * 2 coords
model = backbone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ---- Loss & Optim ----
criterion = nn.SmoothL1Loss(beta=0.01)  # Huber, small beta focuses on L1-like behavior
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# ---- Helpers ----
def mae_per_level(preds, targets):
    preds = preds.reshape(-1,5,2)
    targets = targets.reshape(-1,5,2)
    # absolute error in normalized units
    ae = (preds - targets).abs()
    # mean per level
    return ae.mean(dim=0).mean(dim=1).cpu().numpy()  # (5,)

def evaluate(dl):
    model.eval()
    losses = []
    sum_levels = np.zeros(5, dtype=np.float64)
    count = 0
    with torch.no_grad():
        for x, y, _ in dl:
            x = x.to(device)
            y = y.to(device).float()
            out = model(x)
            loss = criterion(out, y)
            losses.append(loss.item())
            lv = mae_per_level(out, y)
            sum_levels += lv
            count += 1
    return float(np.mean(losses)), (sum_levels / max(1,count))

# ---- Training loop ----
EPOCHS = 15
best_val = 1e9
for epoch in range(1, EPOCHS+1):
    model.train()
    train_losses = []
    for x, y, _ in train_dl:
        x = x.to(device)
        y = y.to(device).float()
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    scheduler.step()

    val_loss, val_mae_lv = evaluate(val_dl)
    print(f"Epoch {epoch:02d} | train {np.mean(train_losses):.4f} | val {val_loss:.4f} | "
          f"MAE (norm) per level {dict(zip(LEVELS, np.round(val_mae_lv,4)))}")

    # checkpoint
    if val_loss < best_val:
        best_val = val_loss
        ckpt_path = OUT_DIR / "resnet18_regression_best.pt"
        torch.save({"model":model.state_dict(),
                    "epoch":epoch,
                    "val_loss":val_loss}, ckpt_path)
        print("  -> saved", ckpt_path)

# ---- Qualitative overlays on VAL set ----
def overlay_and_save(img_path, pred10, gt10, save_path):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    w,h = im.size
    pred = pred10.reshape(5,2)
    gt   = gt10.reshape(5,2)
    # GT: green, Pred: red
    for i,lvl in enumerate(LEVELS):
        px, py = float(pred[i,0])*w, float(pred[i,1])*h
        gx, gy = float(gt[i,0])*w,   float(gt[i,1])*h
        r=3
        draw.ellipse((gx-r,gy-r,gx+r,gy+r), fill=(0,255,0))
        draw.ellipse((px-r,py-r,px+r,py+r), fill=(255,0,0))
        draw.text((gx+4, gy-10), f"{lvl}", fill=(0,255,0))
    im.save(save_path)

model.eval()
VAL_OVL_DIR = OUT_DIR / "val_overlays"
VAL_OVL_DIR.mkdir(exist_ok=True, parents=True)
with torch.no_grad():
    n_saved = 0
    for x, y, fns in val_dl:
        x = x.to(device)
        out = model(x).cpu().numpy()
        y = y.numpy()
        for i in range(len(fns)):
            # find actual path via dataset map
            _, img_path, _ = val_ds.samples[val_ds.samples.index(
                next(s for s in val_ds.samples if s[0]==fns[i])
            )]
            save_path = VAL_OVL_DIR / f"{Path(fns[i]).stem}_overlay.jpg"
            overlay_and_save(img_path, out[i], y[i], save_path)
            n_saved += 1
            if n_saved >= 24:
                break
        if n_saved >= 24:
            break

print("Saved qualitative overlays to:", VAL_OVL_DIR)